<a href="https://colab.research.google.com/github/Hwk040319/MJY-ML-admin/blob/main/test_private.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 배터리 열폭주 이미지 분류 · 비공개 Test 채점 (운영진 전용)

**이 노트북은 운영진 전용입니다.** 참가자에게 공유하지 마세요.

**사용법 요약**: 내 드라이브 어딘가에 `battery-colab-grouped.tar` 사본을 두고, `MJY` 폴더에
팀별 `best_model.pt`를 모아둔 다음, 이 노트북에서 **런타임 → 모두 실행**을 누르세요. 드라이브
연결처럼 한 번만 확인하면 되는 부분만 화면에 뜨는 대로 처리해주면, 나머지는 자동으로 끝까지
실행되면서 `MJY` 폴더 안 모든 팀의 Macro F1이 순서대로 출력됩니다. tar/정답 파일은 드라이브
안 어디에 있든 자동으로 찾습니다 (루트든 `MJY` 폴더 안이든 상관없음).

## 0. GPU 확인

In [ ]:
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
# False 면 런타임 -> 런타임 유형 변경 -> T4 GPU 선택 후 이 셀 다시 실행

## 1. 채점 코드 받기

In [ ]:
!git clone https://github.com/Hwk040319/MJY-ML-admin.git
%cd MJY-ML-admin
!pip install -q -r requirements.txt
print('채점 코드 준비 완료')

## 2. 원본 데이터(비공개 Test 포함) 압축 풀기

실행 전에 구글 드라이브 안 아무 곳에나 `battery-colab-grouped.tar` **사본**을 만들어두세요
(루트든 `MJY` 폴더 안이든 상관없이 아래 셀이 자동으로 찾습니다). 이 셀 실행 중 드라이브 접근
허용 창이 뜨면 허용해주세요 (세션당 한 번만 뜹니다). 14GB라 드라이브에서 먼저 로컬로 복사한
뒤 압축을 푸는데, 복사에만 몇 분 걸릴 수 있습니다 — 도중에 "Input/output error"가 보여도
이 복사 단계가 자동으로 다시 시도하니 기다려주세요.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

tar_matches = list(Path("/content/drive/MyDrive").rglob("battery-colab-grouped.tar"))
if not tar_matches:
    raise FileNotFoundError(
        "battery-colab-grouped.tar를 내 드라이브에서 찾을 수 없습니다. "
        "드라이브에 사본을 만들어뒀는지, 이름이 정확한지 확인하세요.")
if len(tar_matches) > 1:
    tar_matches.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    print(f"주의: tar가 {len(tar_matches)}곳에서 발견되어 가장 최근 파일을 씁니다:")
    for m in tar_matches:
        print(f"  - {m}")
tar_path = tar_matches[0]
print(f"사용할 tar: {tar_path} ({tar_path.stat().st_size / 1e9:.1f} GB)")

# 드라이브에 마운트된 경로에서 직접 tar를 풀면 14GB급 파일에서
# "Cannot read: Input/output error"가 자주 발생합니다 (Drive 스트리밍 마운트의 알려진 한계).
# 그래서 먼저 Colab 로컬 디스크로 복사한 뒤, 로컬 파일을 압축 해제합니다.
local_tar = Path("/content/battery-colab-grouped.tar")
if not local_tar.is_file() or local_tar.stat().st_size != tar_path.stat().st_size:
    for attempt in range(1, 4):
        print(f"드라이브 -> 로컬 복사 시도 {attempt}/3 (몇 분 걸릴 수 있습니다)...")
        !cp "{tar_path}" "{local_tar}"
        if local_tar.is_file() and local_tar.stat().st_size == tar_path.stat().st_size:
            print("복사 완료.")
            break
        print("복사가 덜 됐거나 실패함, 재시도합니다.")
    else:
        raise RuntimeError(
            "드라이브에서 tar 파일을 로컬로 복사하지 못했습니다. "
            "이 셀을 처음부터 다시 실행하거나, 잠시 후 다시 시도하세요.")
else:
    print("이미 로컬에 복사되어 있어 재사용합니다.")

!mkdir -p /content/battery-project-grouped
!tar -xf "{local_tar}" -C /content/battery-project-grouped
!ls /content/battery-project-grouped/participant/data_grouped
# private_test, public_val, train 세 폴더가 보이면 정상입니다.

## 3. 비공개 정답 파일(`private_labels.csv`) 준비

이 파일은 이 저장소에도, 위 tar 안에도 들어있지 않습니다. 내 드라이브 어딘가(예:
`MJY_ML_운영진전용` 폴더)에 `private_labels.csv`라는 이름으로 사본을 넣어두면 아래 셀이
자동으로 찾아서 배치합니다 — 찾지 못하면 업로드 창이 뜨니 그때 골라도 됩니다. (세션당 한 번만
하면 됩니다.)

In [ ]:
from pathlib import Path
from google.colab import files
import shutil

dest = Path("/content/battery-project-grouped/participant/data_grouped/private_test/private_labels.csv")
matches = list(Path("/content/drive/MyDrive").rglob("private_labels.csv"))

if matches:
    if len(matches) > 1:
        matches.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        print(f"주의: private_labels.csv가 {len(matches)}곳에서 발견되어 가장 최근 파일을 씁니다:")
        for m in matches:
            print(f"  - {m}")
    shutil.copy(matches[0], dest)
    print(f"드라이브에서 정답 파일을 찾아 배치했습니다: {matches[0]}")
else:
    uploaded = files.upload()   # 운영진에게 받은 private_labels.csv 선택
    shutil.move(list(uploaded.keys())[0], dest)
    print('업로드한 파일을 배치했습니다.')

## 4. 팀별 Macro F1 한 번에 채점

**미리 준비**: 내 드라이브의 `MJY` 폴더(`/content/drive/MyDrive/MJY`) 안에 팀별 체크포인트를
전부 넣어두세요. 파일 이름은 팀을 구분할 수 있게 자유롭게 지으면 됩니다 (예: `1조.pt`,
`2조_best_model.pt`, 또는 팀별 하위 폴더 `1조/best_model.pt`처럼 넣어도 됩니다 — 하위 폴더까지
전부 찾아서 채점합니다).

이 셀은 그 폴더 안의 `.pt` 파일을 전부 찾아 하나씩 채점합니다. 새 팀 파일을 추가했으면 이 셀만
다시 실행하면 됩니다.

In [ ]:
from pathlib import Path

ckpt_dir = Path("/content/drive/MyDrive/MJY")
ckpt_files = sorted(p for p in ckpt_dir.rglob("*.pt") if p.is_file())

if not ckpt_files:
    print(f"체크포인트(.pt)를 찾을 수 없습니다: {ckpt_dir}")
    print("이 폴더 안에 팀별 best_model.pt 파일을 넣은 뒤 이 셀을 다시 실행하세요.")
else:
    print(f"{len(ckpt_files)}개 체크포인트 발견, 순서대로 채점합니다.\n")
    for ckpt in ckpt_files:
        print(f"\n{'='*20} {ckpt.relative_to(ckpt_dir)} {'='*20}")
        !python score_checkpoint.py \
            --checkpoint "{ckpt}" \
            --data-root /content/battery-project-grouped/participant/data_grouped

## 주의사항

- 최종 순위·등수 계산 방식은 아직 확정 전입니다. 지금은 팀별 `Macro F1` 값만 기록해두세요.
- 채점에 쓰는 데이터(특히 `private_test`, `private_labels.csv`)와 이 노트북·저장소는 참가자에게 절대 공유·유출하면 안 됩니다.
- Colab 세션이 끊기면 1~3번부터 다시 실행하면 됩니다 (드라이브 `MJY` 폴더 내용은 그대로 남아있으니 4번은 바로 다시 돌아갑니다).
- 팀을 나중에 더 추가하고 싶으면, 드라이브 `MJY` 폴더에 파일만 추가하고 4번 셀을 다시 실행하면 됩니다.